# BigAlpha 2026 - Tail-Preserving Pressure Exhaustion

Low-projection companion candidate designed to preserve long-short tail ordering. Direction and all component definitions are frozen from the development sample.

In [ ]:
"""BigAlpha 2026 - champion candidate factor.

The submitted object is one daily factor, but it is built from multiple
independent information channels:

1. late-session five-level order-book pressure;
2. pressure persistence;
3. average queue-size asymmetry;
4. microprice displacement;
5. pressure-exhaustion/price-absorption reversal;
6. a slow quality gate;
7. residualization against the public crowded-factor library.

The final sign is frozen from the 2023 development sample. Higher values mean
stronger expected next-period return.
"""


def _centered_cross_sectional_rank(frame, column):
    """Map a daily cross-sectional percentile rank to [-1, 1]."""
    return (
        frame.groupby("date", sort=False)[column]
        .rank(method="average", pct=True)
        .mul(2.0)
        .sub(1.0)
    )


def _ridge_residualize_by_day(frame, y_column, control_columns, ridge=10.0):
    """Remove same-day crowded-factor projections without using return labels."""
    import numpy as np
    import pandas as pd

    result = frame[y_column].astype("float64").to_numpy(copy=True)
    y_all = frame[y_column].astype("float64").to_numpy()
    x_all = frame[control_columns].astype("float64")

    means = x_all.groupby(frame["date"], sort=False).transform("mean")
    stds = x_all.groupby(frame["date"], sort=False).transform("std")
    x_all = ((x_all - means) / stds.replace(0.0, np.nan)).fillna(0.0)
    x_values = x_all.to_numpy()

    for _, positions in frame.groupby("date", sort=False).indices.items():
        idx = np.asarray(positions, dtype="int64")
        if idx.size < max(30, len(control_columns) * 5):
            continue

        y = y_all[idx]
        x = x_values[idx]
        valid = np.isfinite(y) & np.isfinite(x).all(axis=1)
        if valid.sum() < max(30, len(control_columns) * 5):
            continue

        xv = x[valid]
        yv = y[valid]
        yv = yv - np.nanmean(yv)
        gram = xv.T @ xv
        penalty = np.eye(gram.shape[0], dtype="float64") * float(ridge)
        beta = np.linalg.solve(gram + penalty, xv.T @ yv)

        local = result[idx]
        local[valid] = yv - xv @ beta
        result[idx] = local

    return pd.Series(result, index=frame.index, dtype="float64")


def main(datasources, start_date, end_date):
    """Return exactly: date, instrument, factor."""
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # Two historical trading-day lags are used for stability smoothing.
    # Fourteen natural days safely cover that requirement around holidays.
    buffer_days = 14
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    query_start = start_ts - pd.Timedelta(days=buffer_days)
    query_start_text = query_start.strftime("%Y-%m-%d %H:%M:%S")

    # All calculations below use information available no later than the
    # current trading day. There is no lead, forward join, or future label.
    daily_sql = f"""
    WITH minute_base AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            close,
            (
                coalesce(bid_volume1, 0) + coalesce(bid_volume2, 0) +
                coalesce(bid_volume3, 0) + coalesce(bid_volume4, 0) +
                coalesce(bid_volume5, 0)
            ) AS bid_depth,
            (
                coalesce(ask_volume1, 0) + coalesce(ask_volume2, 0) +
                coalesce(ask_volume3, 0) + coalesce(ask_volume4, 0) +
                coalesce(ask_volume5, 0)
            ) AS ask_depth,
            (
                coalesce(bid_num_orders1, 0) + coalesce(bid_num_orders2, 0) +
                coalesce(bid_num_orders3, 0) + coalesce(bid_num_orders4, 0) +
                coalesce(bid_num_orders5, 0)
            ) AS bid_orders,
            (
                coalesce(ask_num_orders1, 0) + coalesce(ask_num_orders2, 0) +
                coalesce(ask_num_orders3, 0) + coalesce(ask_num_orders4, 0) +
                coalesce(ask_num_orders5, 0)
            ) AS ask_orders,
            bid_price1,
            ask_price1,
            bid_volume1,
            ask_volume1
        FROM {bar1m}
        WHERE close > 0
    ),
    minute_features AS (
        SELECT
            *,
            (bid_depth - ask_depth) /
                nullif(bid_depth + ask_depth, 0) AS depth_imbalance,
            (
                bid_depth / nullif(bid_orders, 0) -
                ask_depth / nullif(ask_orders, 0)
            ) /
            nullif(
                bid_depth / nullif(bid_orders, 0) +
                ask_depth / nullif(ask_orders, 0),
                0
            ) AS order_size_imbalance,
            CASE
                WHEN ask_price1 > bid_price1
                 AND bid_price1 > 0
                 AND coalesce(bid_volume1, 0) + coalesce(ask_volume1, 0) > 0
                THEN 2.0 * (
                    (
                        ask_price1 * coalesce(bid_volume1, 0) +
                        bid_price1 * coalesce(ask_volume1, 0)
                    ) /
                    nullif(coalesce(bid_volume1, 0) + coalesce(ask_volume1, 0), 0)
                    - (ask_price1 + bid_price1) / 2.0
                ) / nullif(ask_price1 - bid_price1, 0)
                ELSE NULL
            END AS microprice_pressure,
            CASE
                WHEN ask_price1 > bid_price1 AND bid_price1 > 0
                THEN (ask_price1 - bid_price1) /
                     nullif((ask_price1 + bid_price1) / 2.0, 0)
                ELSE NULL
            END AS relative_spread,
            close / nullif(
                lag(close) OVER (
                    PARTITION BY instrument, trading_day
                    ORDER BY date
                ),
                0
            ) - 1.0 AS ret_1m
        FROM minute_base
    ),
    daily_all AS (
        SELECT
            trading_day,
            instrument,
            avg(depth_imbalance) AS pressure_all,
            avg(relative_spread) AS relative_spread,
            sqrt(avg(ret_1m * ret_1m)) AS realized_volatility,
            last(close ORDER BY date) /
                nullif(first(close ORDER BY date), 0) - 1.0 AS day_return,
            count(*) AS minute_count
        FROM minute_features
        GROUP BY trading_day, instrument
    ),
    daily_late AS (
        SELECT
            trading_day,
            instrument,
            avg(depth_imbalance) AS pressure_late,
            avg(depth_imbalance * depth_imbalance) AS pressure_late_sq,
            avg(order_size_imbalance) AS order_size_late,
            avg(microprice_pressure) AS microprice_late,
            last(close ORDER BY date) /
                nullif(first(close ORDER BY date), 0) - 1.0 AS late_return,
            count(*) AS late_minute_count
        FROM minute_features
        WHERE strftime(date, '%H:%M') >= '14:30'
          AND strftime(date, '%H:%M') <= '14:56'
        GROUP BY trading_day, instrument
    ),
    daily_close AS (
        SELECT
            trading_day,
            instrument,
            avg(depth_imbalance) AS pressure_close
        FROM minute_features
        WHERE strftime(date, '%H:%M') >= '14:50'
          AND strftime(date, '%H:%M') <= '14:56'
        GROUP BY trading_day, instrument
    )
    SELECT
        CAST(a.trading_day AS DATETIME) AS date,
        a.instrument,
        a.pressure_all,
        l.pressure_late,
        c.pressure_close,
        l.pressure_late /
            nullif(sqrt(l.pressure_late_sq), 0) AS pressure_persistence,
        l.order_size_late,
        l.microprice_late,
        l.late_return,
        a.day_return,
        a.relative_spread,
        a.realized_volatility,
        a.minute_count,
        l.late_minute_count
    FROM daily_all a
    LEFT JOIN daily_late l
      ON a.trading_day = l.trading_day
     AND a.instrument = l.instrument
    LEFT JOIN daily_close c
      ON a.trading_day = c.trading_day
     AND a.instrument = c.instrument
    """

    daily = dai.query(
        daily_sql,
        filters={"date": [query_start_text, end_date]},
        compression=True,
    ).df()

    if daily.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    daily["date"] = pd.to_datetime(daily["date"]).dt.normalize()

    # Rank strictly inside the historical CSI 1000 membership for each day.
    stock_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [query_start_text, end_date]},
        compression=True,
    ).df()
    stock_pool["date"] = pd.to_datetime(stock_pool["date"]).dt.normalize()
    stock_pool = stock_pool.drop_duplicates(["date", "instrument"])
    daily = daily.merge(stock_pool, how="inner", on=["date", "instrument"])

    # Public factors serve two purposes:
    # - slow quality is used only as a nonlinear reliability gate;
    # - crowded technical/flow factors are projected out of the final signal.
    factorlib_columns = [
        "date",
        "instrument",
        "momentum_5",
        "reversal_5",
        "volatility_5",
        "turn",
        "float_market_cap",
        "netflow_amount_rate_main",
        "roe_avg_ttm",
        "net_profit_rate_ttm",
        "debt_to_asset_lf",
    ]
    factorlib_sql = "SELECT " + ", ".join(factorlib_columns) + (
        " FROM bigalpha_2026_factorlib"
    )
    factorlib = dai.query(
        factorlib_sql,
        filters={"date": [query_start_text, end_date]},
        compression=True,
    ).df()
    factorlib["date"] = pd.to_datetime(factorlib["date"]).dt.normalize()
    factorlib = factorlib.drop_duplicates(["date", "instrument"])
    daily = daily.merge(factorlib, how="left", on=["date", "instrument"])

    numeric_columns = [
        "pressure_all",
        "pressure_late",
        "pressure_close",
        "pressure_persistence",
        "order_size_late",
        "microprice_late",
        "late_return",
        "day_return",
        "relative_spread",
        "realized_volatility",
        "minute_count",
        "late_minute_count",
    ] + factorlib_columns[2:]
    for column in numeric_columns:
        daily[column] = pd.to_numeric(daily[column], errors="coerce")
    daily = daily.replace([np.inf, -np.inf], np.nan)

    # Cross-sectional components. Every input is ranked before combination, so
    # quote/volume scale differences cannot dominate the composite.
    daily["late_minus_all"] = (
        daily["pressure_late"] - daily["pressure_all"]
    )
    rank_inputs = [
        "pressure_late",
        "pressure_persistence",
        "order_size_late",
        "microprice_late",
        "late_minus_all",
        "pressure_close",
        "late_return",
        "relative_spread",
        "realized_volatility",
        "roe_avg_ttm",
        "net_profit_rate_ttm",
        "debt_to_asset_lf",
    ]
    for column in rank_inputs:
        daily[f"rank_{column}"] = _centered_cross_sectional_rank(
            daily, column
        ).fillna(0.0)

    # High pressure with a muted contemporaneous price response is interpreted
    # as absorption/hidden accumulation rather than already-realized momentum.
    absorption = (
        daily["rank_pressure_late"]
        - 0.45 * daily["rank_late_return"]
    )
    quality = (
        0.45 * daily["rank_roe_avg_ttm"]
        + 0.35 * daily["rank_net_profit_rate_ttm"]
        - 0.20 * daily["rank_debt_to_asset_lf"]
    )

    raw = (
        0.24 * daily["rank_pressure_late"]
        + 0.16 * daily["rank_pressure_persistence"]
        + 0.15 * daily["rank_order_size_late"]
        + 0.15 * daily["rank_microprice_late"]
        + 0.10 * daily["rank_late_minus_all"]
        + 0.08 * daily["rank_pressure_close"]
        + 0.17 * absorption
        - 0.05 * daily["rank_relative_spread"]
    )
    volatility_gate = (
        1.0 - 0.15 * daily["rank_realized_volatility"].abs()
    ).clip(0.75, 1.0)
    quality_gate = (1.0 + 0.18 * quality).clip(0.75, 1.25)
    daily["factor_raw"] = raw * volatility_gate * quality_gate

    # Short/missing sessions do not receive an artificial extreme signal.
    reliable = (
        daily["minute_count"].fillna(0).ge(180)
        & daily["late_minute_count"].fillna(0).ge(15)
    )
    daily.loc[~reliable, "factor_raw"] = 0.0

    # Three-day causal smoothing raises rolling Elastic Net coefficient
    # stability. Buffer rows are retained until smoothing is complete.
    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)
    grouped = daily.groupby("instrument", sort=False)["factor_raw"]
    lag_1 = grouped.shift(1).fillna(daily["factor_raw"])
    lag_2 = grouped.shift(2).fillna(lag_1)
    daily["factor_smoothed"] = (
        0.60 * daily["factor_raw"] + 0.27 * lag_1 + 0.13 * lag_2
    )

    # Crop only after all causal lag features have consumed the buffer.
    daily = daily[
        (daily["date"] >= start_ts.normalize())
        & (daily["date"] <= end_ts.normalize())
    ].copy()

    daily["log_float_market_cap"] = np.log1p(
        daily["float_market_cap"].clip(lower=0)
    )
    controls = [
        "momentum_5",
        "reversal_5",
        "volatility_5",
        "turn",
        "log_float_market_cap",
        "netflow_amount_rate_main",
        "roe_avg_ttm",
        "net_profit_rate_ttm",
        "debt_to_asset_lf",
    ]
    # The development-period sign is negative: intense late order-book
    # pressure is more consistent with short-horizon liquidity exhaustion than
    # continuation. Freeze that sign before evaluating any later holdout.
    daily["factor"] = -_ridge_residualize_by_day(
        daily,
        y_column="factor_smoothed",
        control_columns=controls,
        ridge=1000000.0,
    )

    daily["factor"] = pd.to_numeric(daily["factor"], errors="coerce")
    daily["factor"] = daily["factor"].replace([np.inf, -np.inf], np.nan)
    day_median = daily.groupby("date", sort=False)["factor"].transform("median")
    daily["factor"] = daily["factor"].fillna(day_median).fillna(0.0)

    result = daily[["date", "instrument", "factor"]].copy()
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
    return result


## Submission contract

`main(datasources, start_date, end_date)` returns exactly `date`, `instrument`, and `factor`. The backward buffer is cropped after causal smoothing; no forward-looking operator is used.